<a href="https://colab.research.google.com/github/jptrainor4-glitch/Data-Analytics/blob/main/Bars_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
'''
Which parts of the Melbourne CBD have strong evening foot traffic but relatively few bar and pub seats,
making them good spots for a new venue?
'''

In [7]:
import pandas as pd
import os
from google.colab import files, drive
import matplotlib.pyplot as plt
import shutil # Import shutil for file operations

drive.mount('/content/drive')                                   #Mount Google Drive

drive_folder = '/content/drive/MyDrive/colab_data/'             # Define the path where you want to save the file in Google Drive

os.makedirs(drive_folder, exist_ok=True)                        # Create the folder if it doesn't exist

expected_filename = 'bars-and-pubs-with-patron-capacity.csv'
drive_filepath = os.path.join(drive_folder, expected_filename)


if os.path.exists(drive_filepath):                              # Check if the file already exists in Google Drive
    print(f"Loading '{expected_filename}' from Google Drive...")
    df = pd.read_csv(drive_filepath)
else:
    print(f"'{expected_filename}' not found in Google Drive. Please upload it.")
    uploaded = files.upload()

    if uploaded:
        filename = list(uploaded.keys())[0]
        df = pd.read_csv(filename)

        shutil.copyfile(filename, drive_filepath)              # Save the uploaded file to Google Drive for future use
        os.remove(filename)                                    # Remove the temporary uploaded file
        print(f"'{filename}' uploaded and saved to Google Drive at '{drive_filepath}'.")
    else:
        print("No file was uploaded. Creating an empty DataFrame.")
        df = pd.DataFrame()                                    # Create an empty DataFrame if no file is uploaded

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading 'bars-and-pubs-with-patron-capacity.csv' from Google Drive...


In [ ]:
df.columns

In [ ]:
#How many rows and columns are there?

'''12 columns, 5304 rows.'''

#What does one row represent? This is the most important question on the list.
#Is it one venue, or something else? Look at the columns and think about why the same bar might show up more than once.
'''One row represents an entry for a specfic venue on a specific census year.
The same bar can show up more than once in the data set as it is recorded year for year.'''


#Which columns are numbers and which are text? Did pandas read them as the right types?

df['Block ID'] = df['Block ID'].astype(str)                     #Convert label information into strings
df['Base property ID'] = df['Base property ID'].astype(str)
df['Property ID'] = df['Property ID'].astype(str)

df.info()

#Where are the missing values, and how many are there?
''' There are 5284 entries for the geographic coordinants, meaning 20 entries are missing this information.'''

#Which years does the data cover, and how many rows are in each year?
print('\n', 'min', df['Census year'].min(), 'max', df['Census year'].max(), '\n')

'''2002 - 2024'''

result = df.groupby('Census year').agg(
    Entry_by_Year=('Census year', 'size'),
    Small_area_count=('CLUE small area', 'nunique')
).reset_index()

print(result, '\n', result.describe())

'''Entries steadly increase through the time period: min - 2002 = 97,
max - 2024 = 304'''

#Is there anything that looks odd, like strange values, duplicates or inconsistent spellings?
df.duplicated(subset=None, keep='first').sum()
df.isna().sum()


In [ ]:
df_missing_rows = df[df.isna().any(axis=1)]             # Collect all rows with NaN values in the data frame and assign to a new DF.

df_missing_rows['Property ID'].isin(df['Property ID'])


In [ ]:
#Looking up missing Geographic values

df_missing_rows = df[df.isna().any(axis=1)]             # Collect all rows with NaN values in the data frame and assign to a new DF.

missing_row_indices = df_missing_rows.index.tolist()    # Convert new missing value dataframe into a python list.

indices_to_display = set()                              # Establish a variable for the set to collect missing value indices, means no duplicated values.

for idx in missing_row_indices:                         # Initiate loop to go through each missing values rows index.
    indices_to_display.add(idx)                         # Add the current missing row index

    if idx > 0:                                         # Add the row before, if it exists
        indices_to_display.add(idx - 1)

    if idx < len(df) - 1:                               # Add the row after, if it exists
        indices_to_display.add(idx + 1)

sorted_indices = sorted(list(indices_to_display))       # Sort the indices for ordered output, convert back into list

display(df.loc[sorted_indices])                         # Select and display the rows from the original DataFrame

In [ ]:
df.columns

In [ ]:
import matplotlib.pyplot as plt

# Create a histogram for 'Number of patrons'
plt.figure(figsize=(10, 6))
plt.hist(df['Number of patrons'].dropna(), bins=100, edgecolor='black')
plt.title('Distribution of Number of Patrons Seating Capacity')
plt.xlabel('Number of Patrons (50s)')
plt.ylabel('Venues')
plt.grid(axis='y', alpha=0.75)
plt.show()

In [ ]:
# Work out arrivals and departures per year from the first and last year for each property, and note the censored years.

df['Trading name'] = df['Trading name'].str.lower()

filtered_df = df.groupby('Trading name').agg(
    Open = ('Census year', min),
    Close = ('Census year', max)
)

print(filtered_df)
print(filtered_df[filtered_df['Open'] == 2002].count())
print(filtered_df[filtered_df['Open'] == 2024].count())

In [70]:
# Finding max unique values of various columns given they are grouped together by property ID

def max_p_id(column_name):    #Takes a column name from sorted_by_id, finds the Property ID with the maximum value
                              #in that column, and returns the rows from the original df DataFrame for that Property ID.
    if column_name not in sorted_by_id.columns:
        print(f"Error: Column '{column_name}' not found in sorted_by_id.")
        return None

    max_property_id = sorted_by_id[column_name].idxmax()        # Get the Property ID with the maximum value in the specified column, this could be altered to find the biggest varience amongest min and max values.
    filtered_df = df[df['Property ID'] == max_property_id]      # Filter the original df DataFrame by this Property ID

    return filtered_df


sorted_by_id = df.groupby('Property ID').agg(                     #New data set that groups each row into the same Property ID
    Trade_Name_count = ('Trading name', 'nunique'),               #Using .agg each column is a count of unique values.
    Business_add_count =('Business address', 'nunique'),          #This will highlight how many different values share the same property id for each column
    Long_count = ('Longitude', 'nunique'),
    Lat_count = ('Latitude', 'nunique'),
    Loc_count = ('location', 'nunique')
)

print(sorted_by_id.describe())                                    #Summerise values

p = max_p_id('Loc_count')                                      #Run function to determine the most unique Location values per proprty ID

Long = 88 * (p['Longitude'].max() - p['Longitude'].min())             #Determine the varience between the max and min location values of given property ID
Lat = 111 * (p['Latitude'].max() - p['Latitude'].min())         #Multipliers are a rough converion from degree into km's

print(Long, Lat)

'''
This particular result finds the most volitale property ID in terms of Location values,
then finds the meter distibution of the min and max location columns.
These results are used to determine wether the spread is too volitial to fill in missing values with
the same propert ID.
'''

       Trade_Name_count  Business_add_count  Long_count   Lat_count  \
count        389.000000          389.000000  389.000000  389.000000   
mean           2.195373            2.503856    2.619537    2.064267   
std            2.239035            2.075411    1.257648    1.014576   
min            1.000000            1.000000    1.000000    1.000000   
25%            1.000000            1.000000    2.000000    1.000000   
50%            2.000000            2.000000    2.000000    2.000000   
75%            3.000000            3.000000    3.000000    2.000000   
max           33.000000           23.000000    9.000000    9.000000   

        Loc_count  
count  389.000000  
mean     2.634961  
std      1.272392  
min      1.000000  
25%      2.000000  
50%      3.000000  
75%      3.000000  
max      9.000000  
0.009010496000200874 0.009037620000086122


'\nThis particular result finds the most volitale property ID in terms of Location values,\nthen finds the meter distibution of the min and max location columns.\nThese results are used to determine wether the spread is too volitial to fill in missing values with\nthe same propert ID.\n'

In [65]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5304 entries, 0 to 5303
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Census year        5304 non-null   int64  
 1   Block ID           5304 non-null   int64  
 2   Property ID        5304 non-null   int64  
 3   Base property ID   5304 non-null   int64  
 4   Building address   5304 non-null   object 
 5   CLUE small area    5304 non-null   object 
 6   Trading name       5304 non-null   object 
 7   Business address   5304 non-null   object 
 8   Number of patrons  5304 non-null   int64  
 9   Longitude          5284 non-null   float64
 10  Latitude           5284 non-null   float64
 11  location           5284 non-null   object 
 12  Missing Location   5304 non-null   bool   
dtypes: bool(1), float64(2), int64(5), object(5)
memory usage: 502.6+ KB
None


In [19]:
#Fill missing values using mean from rows with matching property IDs.

temp = df.groupby('Property ID').agg(                                           #Set up a temp df to hold Long and Lat means for each unique Property ID
    Long = ('Longitude', 'mean'),
    Lat = ('Latitude', 'mean')
)

print(df['Longitude'].isna().sum(), df['Latitude'].isna().sum())                #Check missing values from each column before

df['Longitude'] = df['Longitude'].fillna(df['Property ID'].map(temp['Long']))   #Assign averages to each column using .fillna and .map
df['Latitude'] = df['Latitude'].fillna(df['Property ID'].map(temp['Lat']))

print(df['Longitude'].isna().sum(), df['Latitude'].isna().sum())                #Check missing values after

20 20
0 0


In [ ]:
df_missing_rows = df[df.isna().any(axis=1)]                       #Locating rows with missing values and allocating to a new df

df_cleaned = df.dropna().copy()                                   #Creating a new df excluding the missing values

shared_values = df_missing_rows['Property ID'].isin(df_cleaned['Property ID'])    #Comparing the 2 new dfs to see if they share property IDs

'''
shared_values produces a table which compares the two above df's and creates a boolen.
This determines wether the rows in the missing_rows df share Property ID with rows in the cleaned df.
'''
if len(df_missing_rows) > 0:
  df['Missing Location'] = df.isna().any(axis=1)              #Add a new column to df that marks the missing values rows

In [ ]:
print(df[['Trading name', 'Number of patrons']].count(), '\n')

df_bar = df.groupby('Trading name')[['Census year', 'Number of patrons']]     #new data frame = Grouping the og data frame by commodity and then adding a revenue column by summing each price witihn a particular group

print(df_bar.count())